# Exploratory Data Analysis

This notebook is the first step in the Seattle building energy case study. It introduces the raw dataset, inspects column structure, evaluates missing values, and reviews the semantic role of the main fields before any cleaning or modeling.

## Objectives

- Inspect the raw Seattle energy dataset.
- Identify key columns for prediction and feature engineering.
- Check missing data, categorical variables, and compliance metadata.
- Save a filtered dataset for use in the preparation notebook.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
import os
import numpy as np
from pathlib import Path

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gensim 4.3.0 requires FuzzyTM>=0.4.0, which is not installed.
langchain 0.0.342 requires numpy<2,>=1, but you have numpy 2.4.4 which is incompatible.
radcad 0.8.4 requires pandas<2.0.0,>=1.0.0, but you have pandas 3.0.2 which is incompatible.
scipy 1.11.3 requires numpy<1.28.0,>=1.21.6, but you have numpy 2.4.4 which is incompatible.


Note: you may need to restart the kernel to use updated packages.



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\karap\anaconda3\envs\datascience1\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\Users\karap\anaconda3\envs\datascience1\Lib\site-packages\traitlets\config\application.py", line 992, in launch_instance
    app.start()
  File "c:\Users\karap\anaconda3\envs\datascience1\Lib\site-packages\ipykernel\kernelapp.py", line 736, in start
    self.io_loop.s

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\karap\anaconda3\envs\datascience1\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\Users\karap\anaconda3\envs\datascience1\Lib\site-packages\traitlets\config\application.py", line 992, in launch_instance
    app.start()
  File "c:\Users\karap\anaconda3\envs\datascience1\Lib\site-packages\ipykernel\kernelapp.py", line 736, in start
    self.io_loop.s

AttributeError: _ARRAY_API not found

ImportError: DLL load failed while importing _imaging: Le module spécifié est introuvable.

## Load the raw dataset

In [ ]:
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

input_file = project_root / "data" / "raw" / "2016_Building_Energy_Benchmarking.csv"
df = pd.read_csv(input_file)

# Display column names to identify the usage column
print("Available columns:", df.columns.tolist())

print("Raw dataset loaded from:", input_file)
print("Dimensions:", df.shape)
print("Columns:", len(df.columns))

## Selection of Non-Residential Buildings

The goal is to restrict the dataset to building types relevant for modeling energy consumption in non-residential contexts and major campuses.

In [ ]:
# Here are the values to keep:
non_residential_keywords = [
    "NonResidential",
    "Nonresidential COS",
    "Nonresidential WA",
    "SPS-District K-12",
    "Campus"
]

# Adapt here with the actual usage column name
candidate_columns = [
    "BuildingType",
    "PrimaryPropertyType",
    "LargestPropertyUseType",
    "ListOfAllPropertyUseTypes"
]
usage_column = next((col for col in candidate_columns if col in df.columns), None)

if usage_column is None:
    raise ValueError("No building type column found in the raw dataset.")

print("Using building type column:", usage_column)

# Filter rows containing the specified types
filtered_df = df[df[usage_column].isin(non_residential_keywords)]

print("Filtered rows:", filtered_df.shape[0], "/", df.shape[0])
print("Retained portion:", filtered_df.shape[0] / len(df))

In [ ]:
output_file = project_root / "data" / "processed" / "2016_Building_Energy_Benchmarking_Purge.csv"
output_file.parent.mkdir(parents=True, exist_ok=True)

# Save the result to a new CSV file
filtered_df.to_csv(output_file, index=False)

print(f"Filtered file saved to: {output_file}")

## General Overview and Missing Values

Examine data types and detect columns with missing values before cleaning.

In [ ]:
# 1. Loading data and general overview
purge_file = project_root / "data" / "processed" / "2016_Building_Energy_Benchmarking_Purge.csv"
building_consumption = pd.read_csv(purge_file)
# General overview
print("Dimensions:", building_consumption.shape)
building_consumption.head(5)

### Data type info - null and non-null counts

In [ ]:
# 2. General column information
# Info on data types and null/non-null counts
building_consumption.info()


In [ ]:
# 3. Missing values analysis for building_consumption
# Count and percentage of missing values per column
missing_data = building_consumption.isnull().sum().to_frame('MissingCount')
missing_data['MissingPct'] = 100 * missing_data['MissingCount'] / len(building_consumption)
missing_data = missing_data[missing_data['MissingCount'] > 0].sort_values(by='MissingPct', ascending=False)

missing_data

### Categorical variable analysis

In [ ]:
# 4. Categorical variable analysis

# Object-type columns (categorical or text)
cat_cols = building_consumption.select_dtypes(include='object').columns.tolist()

# Unique counts per categorical variable
building_consumption[cat_cols].nunique().sort_values(ascending=False)

## Semantic review of key columns

This table summarizes the main column groups and their likely role in the modeling pipeline.

| Column | Role | Notes |
| --- | --- | --- |
| `SiteEnergyUse(kBtu)` | Target variable | Main energy consumption target for regression. |
| `PropertyGFATotal` | Building size | Strong predictor of energy use. |
| `YearBuilt` | Building age | Useful as a physical descriptor. |
| `PrimaryPropertyType` | Building usage | Strong semantic feature. |
| `LargestPropertyUseType` | Usage category | May overlap with `PrimaryPropertyType` but still useful. |
| `ComplianceStatus` | Data quality filter | Use it to keep only reliable records. |
| `TotalGHGEmissions` | Emissions outcome | Related to energy use and can support domain analysis. |

## Compliance status analysis

Review the distribution of compliance statuses to decide whether to restrict the dataset to records with reliable reporting.

In [ ]:
if 'ComplianceStatus' in filtered_df.columns:
    counts = filtered_df['ComplianceStatus'].value_counts(dropna=False)
    pct = filtered_df['ComplianceStatus'].value_counts(normalize=True, dropna=False) * 100
    compliance_summary = pd.DataFrame({
        'count': counts,
        'percent': pct.round(2)
    })
    display(compliance_summary)
else:
    print('ComplianceStatus column not found in this dataset.')

## Summary and next step

This exploratory notebook is meant to give students a first look at the raw dataset structure and the main semantic groups. The next notebook, `01_clean_data.ipynb`, uses this understanding to perform data cleaning, deduplication, and outlier detection before feature engineering.